In [1]:
#author: Avery
#The point of this script is to associate the salmon, prokka, and kofamscan outputs with each other
#so that we can associate the contig IDs and associated number of read hits to each contig
#with the locus IDs and annotations from prokka and kofamscan. I do this by converting the output
#files to pandas dataframes and merging the dataframes together.
#In this script, I also get the average abundance of single-copy COGs and the total number of read 
#hits to metatranscriptomic samples, and I divide the metagenomic and metatranscriptomic read hits
#to genes of interest to these values, respectively. The output is two tsv files (for metaG and metaT)
#with the standardized abundance of all genes of interest specified in the paper.

In [8]:
#import dependencies
import pandas as pd
import os

In [9]:
#parse gff file to get locus IDs

#make dictionary where keys are contig names and values are lists of one or more locus IDs
contig_locusid = {}
index = 0

with open("../output/data_processing/prokka-contigs-idsonly.gff") as prodigal_gff: #open gff file
    for line in prodigal_gff:
        l = line.split() #for each line, make list of values separated by whitespace
        if l[0] in contig_locusid.keys(): #if contig name already in the dictionary
            contig_locusid[l[0]].append(l[8].split(';')[0][3:]) #append the locus ID to the value list
        if l[0] not in contig_locusid.keys(): #if contig name not already in dictionary
            contig_locusid[l[0]] = [] #add contig name as a key and empty list as a value in dictionary
            contig_locusid[l[0]].append(l[8].split(';')[0][3:]) #append the locus ID to the value list
        index += 1 #count number of lines in file
prodigal_gff.close() #close gff file

print(l)
print(l[8].split(';')[0][3:])
print(index) #print number of lines in file

['k141_151157_FS856', 'Prodigal:002006', 'CDS', '120', '845', '.', '+', '0', 'ID=HHHBGIEE_666767;inference=ab', 'initio', 'prediction:Prodigal:002006;locus_tag=HHHBGIEE_666767;product=hypothetical', 'protein']
HHHBGIEE_666767
666767


In [10]:
#check that all locus IDs are included in dictionary by making sure number of locus IDs = number of lines in file
index = 0
for lst in contig_locusid.values():
    index += len(lst)
print(index)

666767


In [11]:
#whoops. create a dictionary where keys are locus IDs and values are contig names. the opposite of above!
locusid_contig = {}
index = 0

with open("../output/data_processing/prokka-contigs-idsonly.gff") as prodigal_gff: #open gff file
    for line in prodigal_gff:
        l = line.split() #for each line, make list of values separated by whitespace
        locusid_contig[l[8].split(';')[0][3:]] = l[0]
        index += 1
prodigal_gff.close()

print(index)

666767


In [12]:
#check that all locus IDs are included in dictionary by making sure number of locus IDs = number of lines in file
print(len(locusid_contig.keys()))

666767


In [13]:
#make locusid_contig into a dataframe with two columns, locusid and contig

df_locusid_contig = pd.DataFrame({'locusid' : locusid_contig.keys(),
                                'contig' : locusid_contig.values(),}, 
                                columns=['locusid','contig'])

print(df_locusid_contig)

                locusid             contig
0        HHHBGIEE_00001  k141_115416_FS851
1        HHHBGIEE_00002  k141_115416_FS851
2        HHHBGIEE_00003  k141_217079_FS851
3        HHHBGIEE_00004  k141_195103_FS851
4        HHHBGIEE_00005  k141_186862_FS851
...                 ...                ...
666762  HHHBGIEE_666763  k141_151153_FS856
666763  HHHBGIEE_666764  k141_151155_FS856
666764  HHHBGIEE_666765  k141_151155_FS856
666765  HHHBGIEE_666766  k141_151156_FS856
666766  HHHBGIEE_666767  k141_151157_FS856

[666767 rows x 2 columns]


In [15]:
#turn each quant.sf file into a dataframe
df_mG_FS851 = pd.read_csv('../output/data_processing/mG_FS851_quant.sf',sep = '\t')
df_mG_FS852 = pd.read_csv('../output/data_processing/mG_FS852_quant.sf',sep = '\t')
df_mG_FS854 = pd.read_csv('../output/data_processing/mG_FS854_quant.sf',sep = '\t')
df_mG_FS856 = pd.read_csv('../output/data_processing/mG_FS856_quant.sf',sep = '\t')
df_mT_FS851 = pd.read_csv('../output/data_processing/mT_FS851_quant.sf',sep = '\t')
df_mT_FS852 = pd.read_csv('../output/data_processing/mT_FS852_quant.sf',sep = '\t')
df_mT_FS854 = pd.read_csv('../output/data_processing/mT_FS854_quant.sf',sep = '\t')
df_mT_FS856 = pd.read_csv('../output/data_processing/mT_FS856_quant.sf',sep = '\t')

print(df_mG_FS851)

                     Name  Length  EffectiveLength       TPM  NumReads
0        k141_60456_FS851     349           99.004  5.754657       1.0
1       k141_145644_FS851     464          214.000  2.662299       1.0
2       k141_167628_FS851     558          308.000  1.849779       1.0
3        k141_10992_FS851     403          153.000  3.723738       1.0
4       k141_115416_FS851     698          448.000  1.271723       1.0
...                   ...     ...              ...       ...       ...
857861  k141_151154_FS856     444          194.000  0.000000       0.0
857862  k141_151155_FS856    1541         1291.000  0.000000       0.0
857863  k141_151156_FS856    1581         1331.000  0.000000       0.0
857864  k141_151157_FS856    1051          801.000  0.000000       0.0
857865  k141_151158_FS856     418          168.000  0.000000       0.0

[857866 rows x 5 columns]


In [16]:
#drop unnecessary columns
df_mG_FS851 = df_mG_FS851.drop(columns=['Length','EffectiveLength','TPM'])
df_mG_FS852 = df_mG_FS852.drop(columns=['Length','EffectiveLength','TPM'])
df_mG_FS854 = df_mG_FS854.drop(columns=['Length','EffectiveLength','TPM'])
df_mG_FS856 = df_mG_FS856.drop(columns=['Length','EffectiveLength','TPM'])
df_mT_FS851 = df_mT_FS851.drop(columns=['Length','EffectiveLength','TPM'])
df_mT_FS852 = df_mT_FS852.drop(columns=['Length','EffectiveLength','TPM'])
df_mT_FS854 = df_mT_FS854.drop(columns=['Length','EffectiveLength','TPM'])
df_mT_FS856 = df_mT_FS856.drop(columns=['Length','EffectiveLength','TPM'])

In [17]:
#rename NumReads to be sample-specific
df_mG_FS851 = df_mG_FS851.rename(columns={"NumReads": "mG_FS851_NumReads"})
df_mG_FS852 = df_mG_FS852.rename(columns={"NumReads": "mG_FS852_NumReads"})
df_mG_FS854 = df_mG_FS854.rename(columns={"NumReads": "mG_FS854_NumReads"})
df_mG_FS856 = df_mG_FS856.rename(columns={"NumReads": "mG_FS856_NumReads"})
df_mT_FS851 = df_mT_FS851.rename(columns={"NumReads": "mT_FS851_NumReads"})
df_mT_FS852 = df_mT_FS852.rename(columns={"NumReads": "mT_FS852_NumReads"})
df_mT_FS854 = df_mT_FS854.rename(columns={"NumReads": "mT_FS854_NumReads"})
df_mT_FS856 = df_mT_FS856.rename(columns={"NumReads": "mT_FS856_NumReads"})

print(df_mG_FS851)
print(df_mG_FS856)
print(df_mT_FS852)

                     Name  mG_FS851_NumReads
0        k141_60456_FS851                1.0
1       k141_145644_FS851                1.0
2       k141_167628_FS851                1.0
3        k141_10992_FS851                1.0
4       k141_115416_FS851                1.0
...                   ...                ...
857861  k141_151154_FS856                0.0
857862  k141_151155_FS856                0.0
857863  k141_151156_FS856                0.0
857864  k141_151157_FS856                0.0
857865  k141_151158_FS856                0.0

[857866 rows x 2 columns]
                     Name  mG_FS856_NumReads
0        k141_60456_FS851                0.0
1       k141_145644_FS851                0.0
2       k141_167628_FS851                0.0
3        k141_10992_FS851                0.0
4       k141_115416_FS851                0.0
...                   ...                ...
857861  k141_151154_FS856                1.0
857862  k141_151155_FS856                1.0
857863  k141_151156_FS856   

In [18]:
#merge all dataframes to get contig names + sample ID as columns

df_numreads = pd.concat([df_mG_FS851, df_mG_FS852[['mG_FS852_NumReads']], 
                        df_mG_FS854[['mG_FS854_NumReads']], df_mG_FS856[['mG_FS856_NumReads']], 
                        df_mT_FS851[['mT_FS851_NumReads']], df_mT_FS852[['mT_FS852_NumReads']],
                        df_mT_FS854[['mT_FS854_NumReads']], df_mT_FS856[['mT_FS856_NumReads']]],
                      axis="columns")

df_numreads = df_numreads.rename(columns={"Name": "contig"})

print(df_numreads)

                   contig  mG_FS851_NumReads  mG_FS852_NumReads  \
0        k141_60456_FS851                1.0                0.0   
1       k141_145644_FS851                1.0                0.0   
2       k141_167628_FS851                1.0                0.0   
3        k141_10992_FS851                1.0                0.0   
4       k141_115416_FS851                1.0                0.0   
...                   ...                ...                ...   
857861  k141_151154_FS856                0.0                0.0   
857862  k141_151155_FS856                0.0                0.0   
857863  k141_151156_FS856                0.0                0.0   
857864  k141_151157_FS856                0.0                0.0   
857865  k141_151158_FS856                0.0                0.0   

        mG_FS854_NumReads  mG_FS856_NumReads  mT_FS851_NumReads  \
0                     0.0                0.0                0.0   
1                     0.0                0.0                7

In [19]:
#merge dataframe with sample names with contig and locus tag dataframe
df_contig_locusid_numreads = df_numreads.merge(df_locusid_contig, on="contig", how="outer")

print(df_contig_locusid_numreads)

                    contig  mG_FS851_NumReads  mG_FS852_NumReads  \
0         k141_60456_FS851                1.0                0.0   
1        k141_145644_FS851                1.0                0.0   
2        k141_167628_FS851                1.0                0.0   
3         k141_10992_FS851                1.0                0.0   
4        k141_115416_FS851                1.0                0.0   
...                    ...                ...                ...   
1111281  k141_151155_FS856                0.0                0.0   
1111282  k141_151155_FS856                0.0                0.0   
1111283  k141_151156_FS856                0.0                0.0   
1111284  k141_151157_FS856                0.0                0.0   
1111285  k141_151158_FS856                0.0                0.0   

         mG_FS854_NumReads  mG_FS856_NumReads  mT_FS851_NumReads  \
0                      0.0                0.0                0.0   
1                      0.0                0.0  

In [20]:
#get COG annotations from prodigal output into dataframe
df_COGs = pd.read_csv('../output/data_processing/all-contigs-prokkannotated.tsv',sep = '\t')
#drop unnecessary columns
df_COGs = df_COGs.drop(columns=['ftype','length_bp','EC_number'])
#standardize column names
df_COGs = df_COGs.rename(columns={"gene" : "COG_gene"})
df_COGs = df_COGs.rename(columns={"locus_tag" : "locusid"})

#merge COGs dataframe with the dataframe of contigs and locus ids from above to associate these two datasets
df_COGs_numreads = df_contig_locusid_numreads.merge(df_COGs, on="locusid", how="outer")

print(df_COGs_numreads)

                     contig  mG_FS851_NumReads  mG_FS852_NumReads  \
0          k141_60456_FS851                1.0                0.0   
1          k141_60456_FS851                1.0                0.0   
2          k141_60456_FS851                1.0                0.0   
3          k141_60456_FS851                1.0                0.0   
4          k141_60456_FS851                1.0                0.0   
...                     ...                ...                ...   
42896067  k141_151153_FS856                0.0                0.0   
42896068  k141_151155_FS856                0.0                0.0   
42896069  k141_151155_FS856                0.0                0.0   
42896070  k141_151156_FS856                0.0                0.0   
42896071  k141_151157_FS856                0.0                0.0   

          mG_FS854_NumReads  mG_FS856_NumReads  mT_FS851_NumReads  \
0                       0.0                0.0                0.0   
1                       0.0      

In [13]:
#get KEGG annotations from kofamscan output into dataframe
index = 0
gene_name = []
KO = []
KO_definition = []

with open("../output/data_processing/all.contigs_kofam.txt") as kofam:
    for line in kofam:
        if index > 1:
            if '*' in line:
                gene_name.append(line.split()[1])
                KO.append(line.split()[2])
                KO_definition.append(' '.join(line.split()[6:]))
            if '*' not in line:
                gene_name.append(line.split()[0])
                KO.append(line.split()[1])
                KO_definition.append(' '.join(line.split()[5:]))
        index += 1
kofam.close()

In [14]:
data_KEGGs = {'locusid': gene_name, 'KO': KO, 'KO_Definition': KO_definition}
df_KEGGs = pd.DataFrame(data=data_KEGGs)
print(df_KEGGs)

                 locusid      KO  \
0         HHHBGIEE_00002  K01372   
1         HHHBGIEE_00002  K26834   
2         HHHBGIEE_00002  K01363   
3         HHHBGIEE_00002  K16290   
4         HHHBGIEE_00009  K02194   
...                  ...     ...   
7138232  HHHBGIEE_666766  K01552   
7138233  HHHBGIEE_666767  K07453   
7138234  HHHBGIEE_666767  K07451   
7138235  HHHBGIEE_666767  K07454   
7138236  HHHBGIEE_666767  K12212   

                                             KO_Definition  
0                       bleomycin hydrolase [EC:3.4.22.40]  
1            Marseillevirus papain-like cysteine peptidase  
2                                cathepsin B [EC:3.4.22.1]  
3                  xylem cysteine proteinase [EC:3.4.22.-]  
4                                  heme exporter protein B  
...                                                    ...  
7138232  energy-coupling factor transport system ATP-bi...  
7138233                  putative restriction endonuclease  
7138234  5-methylc

In [15]:
#merge locus id, contig, salmon info from each sample, KO annotations, COG annotations
df_all = df_COGs_numreads.merge(df_KEGGs, on="locusid", how="outer")

In [16]:
#output df_all to tsv for safekeeping
print(df_all)
#df_all.to_csv('concatenateddf.tsv', sep="\t")

                     contig  mG_FS851_NumReads  mG_FS852_NumReads  \
0          k141_60456_FS851                1.0                0.0   
1          k141_60456_FS851                1.0                0.0   
2          k141_60456_FS851                1.0                0.0   
3          k141_60456_FS851                1.0                0.0   
4          k141_60456_FS851                1.0                0.0   
...                     ...                ...                ...   
49664293  k141_151156_FS856                0.0                0.0   
49664294  k141_151157_FS856                0.0                0.0   
49664295  k141_151157_FS856                0.0                0.0   
49664296  k141_151157_FS856                0.0                0.0   
49664297  k141_151157_FS856                0.0                0.0   

          mG_FS854_NumReads  mG_FS856_NumReads  mT_FS851_NumReads  \
0                       0.0                0.0                0.0   
1                       0.0      

In [21]:
#get list of single copy cog IDs from text file
singlecopycogs = []

with open('../output/data_processing/singlecopycogs.txt') as f:
    for line in f:
        l = line.strip()
        singlecopycogs.append(l)
f.close()
print(singlecopycogs)

['COG0012', 'COG0016', 'COG0048', 'COG0049', 'COG0052', 'COG0080', 'COG0081', 'COG0085', 'COG0087', 'COG0088', 'COG0090', 'COG0091', 'COG0092', 'COG0093', 'COG0094', 'COG0096', 'COG0097', 'COG0098', 'COG0099', 'COG0100', 'COG0102', 'COG0103', 'COG0124', 'COG0184', 'COG0185', 'COG0186', 'COG0197', 'COG0200', 'COG0201', 'COG0256', 'COG0495', 'COG0522', 'COG0525', 'COG0533', 'COG0541']


In [18]:
#for each cogid, get the sum of the read hits to single-copy COGs per metagenomic sample
mG_FS851_sccog_sum = []
mG_FS852_sccog_sum = []
mG_FS854_sccog_sum = []
mG_FS856_sccog_sum = []

for cogid in singlecopycogs:
    mG_FS851_sccog_sum.append(sum(df_all[df_all['COG'] == cogid]['mG_FS851_NumReads'].tolist()))
    mG_FS852_sccog_sum.append(sum(df_all[df_all['COG'] == cogid]['mG_FS852_NumReads'].tolist()))
    mG_FS854_sccog_sum.append(sum(df_all[df_all['COG'] == cogid]['mG_FS854_NumReads'].tolist()))
    mG_FS856_sccog_sum.append(sum(df_all[df_all['COG'] == cogid]['mG_FS856_NumReads'].tolist()))

#put these sums in a dataframe with columns as sample name and single-copy COG id as rows
d = {'cogid': singlecopycogs, 
    'mG_FS851_NumReads': mG_FS851_sccog_sum,
    'mG_FS852_NumReads': mG_FS852_sccog_sum,
    'mG_FS854_NumReads': mG_FS854_sccog_sum,
    'mG_FS856_NumReads': mG_FS856_sccog_sum}
df_sccogs = pd.DataFrame(data=d)

In [19]:
#output sum of read hits to single copy COGs per metagenomic sample to tsv for safekeeping
print(df_sccogs)
#df_sccogs.to_csv('sccogs_readhits.tsv', sep="\t")

      cogid  mG_FS851_NumReads  mG_FS852_NumReads  mG_FS854_NumReads  \
0   COG0012              142.0              213.0             1040.0   
1   COG0016               33.0               46.0              198.0   
2   COG0048               31.0               25.0               15.0   
3   COG0049               48.0               49.0               68.0   
4   COG0052                8.0               27.0              191.0   
5   COG0080               24.0               29.0               54.0   
6   COG0081               27.0               28.0              119.0   
7   COG0085              105.0              176.0              957.0   
8   COG0087               16.0               28.0               86.0   
9   COG0088                8.0                8.0               70.0   
10  COG0090               34.0               36.0               72.0   
11  COG0091               13.0               22.0               83.0   
12  COG0092               34.0               34.0              2

In [20]:
#create dictionary of average single copy COG read hits per sample
sccog_dict = {'mG_FS851_sccogs_avg': sum(mG_FS851_sccog_sum)/len(mG_FS851_sccog_sum),
              'mG_FS852_sccogs_avg': sum(mG_FS852_sccog_sum)/len(mG_FS852_sccog_sum),
              'mG_FS854_sccogs_avg': sum(mG_FS854_sccog_sum)/len(mG_FS854_sccog_sum),
              'mG_FS856_sccogs_avg': sum(mG_FS856_sccog_sum)/len(mG_FS856_sccog_sum)}
print(sccog_dict)

{'mG_FS851_sccogs_avg': 48.77142857142857, 'mG_FS852_sccogs_avg': 56.0, 'mG_FS854_sccogs_avg': 237.94285714285715, 'mG_FS856_sccogs_avg': 180.6}


In [21]:
#create dictionary of total number of metatranscriptomic reads per sample
mT_readhits_dict = {'mT_FS851_readhits': sum(df_all['mT_FS851_NumReads'].tolist()),
              'mT_FS852_readhits': sum(df_all['mT_FS852_NumReads'].tolist()),
              'mT_FS854_readhits': sum(df_all['mT_FS854_NumReads'].tolist()),
              'mT_FS856_readhits': sum(df_all['mT_FS856_NumReads'].tolist())}
print(mT_readhits_dict)

{'mT_FS851_readhits': 436525353.0, 'mT_FS852_readhits': 1915431988.0, 'mT_FS854_readhits': 282876734.0, 'mT_FS856_readhits': 812137068.0}


In [22]:
#get a list of each gene of interest, followed by the KO ids that correspond to the gene
genesofinterest = []

with open('../output/data_processing/genesofinterest.txt') as f:
    for line in f:
        l = line.strip().split(',')
        genesofinterest.append(l)
f.close()

print(genesofinterest)

[['ech hydrogenase (echABCE)', 'K14086', 'K14087', 'K14088', 'K14089', 'K14090'], ['energy-converting hydrogenase A (ehaBCEGNOP)', 'K14093', 'K14094', 'K14096', 'K14098', 'K14105', 'K14106', 'K14107'], ['energy-converting hydrogenase B (ehbAHFIJKLNO)', 'K14092', 'K14093', 'K14097', 'K14100', 'K14101', 'K14102', 'K14103', 'K14105', 'K14106'], ['coenzyme F420 hydrogenase (frhABDG)', 'K00440', 'K00441', 'K00442', 'K00443'], ['hydrogenase (hyaABC/hybO/hybC)', 'K06282', 'K06281', 'K03620', 'K06282', 'K06281'], ['[NiFe] hydrogenase (hydAB)', 'K18008', 'K00437'], ['methane/ammonia monooxygenase (pmoA-amoA)', 'K10944'], ['methyl-coenzyme M reductase (mcrABCDG)', 'K00399', 'K00401', 'K03421', 'K03422', 'K00402'], ['nitrous-oxide reductase (nosZ)', 'K00376'], ['nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)', 'K00368', 'K15864'], ['nitric oxide reductase (norB)', 'K04561'], ['nitrite reductase (cytochrome c-552) (nrfA)', 'K03385'], ['cytochrome c oxidase (coxABCD/AC/ctaF)', 'K022

In [34]:
#print for easy copy-pasting
for lst in genesofinterest:
    print(lst[0] + ':0,')

ech hydrogenase (echABCE):0,
energy-converting hydrogenase A (ehaBCEGNOP):0,
energy-converting hydrogenase B (ehbAHFIJKLNO):0,
coenzyme F420 hydrogenase (frhABDG):0,
hydrogenase (hyaABC/hybO/hybC):0,
[NiFe] hydrogenase (hydAB):0,
methane/ammonia monooxygenase (pmoA-amoA):0,
methyl-coenzyme M reductase (mcrABCDG):0,
nitrous-oxide reductase (nosZ):0,
nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS):0,
nitric oxide reductase (norB):0,
nitrite reductase (cytochrome c-552) (nrfA):0,
cytochrome c oxidase (coxABCD/AC/ctaF):0,
cytochrome c oxidase cbb3-type (ccoNOPQ_NO):0,
carbon-monoxide dehydrogenase (cooCFS/acsA):0,
ATP-citrate lyase (aclAB):0,
fumarate reductase (frdAB):0,
2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD):0,
sulfite reductase (dsrAB):0,
sulfate adenylyltransferase (sat/met3):0,
sulfur-oxidizing protein Sox (soxABCXYZ):0,
sulfide:quinone oxidoreductase (sqr):0,
iron transport protein (FeoB/C):0,
iron transport protein (FeoA):0,


In [35]:
#create dictionaries with set keys so I can += each abundance more easily in the coming for loop
mG_FS851_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

mG_FS852_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

mG_FS854_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

mG_FS856_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

mT_FS851_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

mT_FS852_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

mT_FS854_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

mT_FS856_genereads = {'ech hydrogenase (echABCE)':0,
'energy-converting hydrogenase A (ehaBCEGNOP)':0,
'energy-converting hydrogenase B (ehbAHFIJKLNO)':0,
'coenzyme F420 hydrogenase (frhABDG)':0,
'hydrogenase (hyaABC/hybO/hybC)':0,
'[NiFe] hydrogenase (hydAB)':0,
'methane/ammonia monooxygenase (pmoA-amoA)':0,
'methyl-coenzyme M reductase (mcrABCDG)':0,
'nitrous-oxide reductase (nosZ)':0,
'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)':0,
'nitric oxide reductase (norB)':0,
'nitrite reductase (cytochrome c-552) (nrfA)':0,
'cytochrome c oxidase (coxABCD/AC/ctaF)':0,
'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)':0,
'carbon-monoxide dehydrogenase (cooCFS/acsA)':0,
'ATP-citrate lyase (aclAB)':0,
'fumarate reductase (frdAB)':0,
'2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)':0,
'sulfite reductase (dsrAB)':0,
'sulfate adenylyltransferase (sat/met3)':0,
'sulfur-oxidizing protein Sox (soxABCXYZ)':0,
'sulfide:quinone oxidoreductase (sqr)':0,
'iron transport protein (FeoB/C)':0,
'iron transport protein (FeoA)':0}

In [37]:
#separate out the KO IDs from genes of interest for mG samples and mT samples separately
for lst in genesofinterest:
    for KOid in lst[1:]:
        #with the gene group key of lst[0], add the sum of all read hits to a certain KO id to the dictionary value
        mG_FS851_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mG_FS851_NumReads'].tolist())
        mG_FS852_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mG_FS852_NumReads'].tolist())
        mG_FS854_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mG_FS854_NumReads'].tolist())
        mG_FS856_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mG_FS856_NumReads'].tolist())
        mT_FS851_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mT_FS851_NumReads'].tolist())
        mT_FS852_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mT_FS852_NumReads'].tolist())
        mT_FS854_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mT_FS854_NumReads'].tolist())
        mT_FS856_genereads[lst[0]] += sum(df_all[df_all['KO'] == KOid]['mT_FS856_NumReads'].tolist())

print(mT_FS852_genereads)

{'ech hydrogenase (echABCE)': 340946.0, 'energy-converting hydrogenase A (ehaBCEGNOP)': 842943.0, 'energy-converting hydrogenase B (ehbAHFIJKLNO)': 294356.0, 'coenzyme F420 hydrogenase (frhABDG)': 1306331.0, 'hydrogenase (hyaABC/hybO/hybC)': 503106.0, '[NiFe] hydrogenase (hydAB)': 171342.0, 'methane/ammonia monooxygenase (pmoA-amoA)': 23.0, 'methyl-coenzyme M reductase (mcrABCDG)': 142682.0, 'nitrous-oxide reductase (nosZ)': 16234.0, 'nitrite reductase (NO-forming)/hydroxylamine reductase (nirKS)': 81117.0, 'nitric oxide reductase (norB)': 50165.0, 'nitrite reductase (cytochrome c-552) (nrfA)': 48924.0, 'cytochrome c oxidase (coxABCD/AC/ctaF)': 261302.0, 'cytochrome c oxidase cbb3-type (ccoNOPQ_NO)': 444630.0, 'carbon-monoxide dehydrogenase (cooCFS/acsA)': 1182726.0, 'ATP-citrate lyase (aclAB)': 423663.0, 'fumarate reductase (frdAB)': 446593.0, '2-oxoglutarate/2-oxoacid ferredoxin oxidoreductase (korABCD/oorABCD)': 1071435.0, 'sulfite reductase (dsrAB)': 492856.0, 'sulfate adenylyltran

In [40]:
#put into dataframe
d = {'gene_group': mG_FS851_genereads.keys(), 
    'mG_FS851': mG_FS851_genereads.values(),
    'mG_FS852': mG_FS852_genereads.values(),
    'mG_FS854': mG_FS854_genereads.values(),
    'mG_FS856': mG_FS856_genereads.values()}
df_mG_genesofinterest = pd.DataFrame(data=d)

d2 = {'gene_group': mT_FS851_genereads.keys(), 
    'mT_FS851': mT_FS851_genereads.values(),
    'mT_FS852': mT_FS852_genereads.values(),
    'mT_FS854': mT_FS854_genereads.values(),
    'mT_FS856': mT_FS856_genereads.values()}
df_mT_genesofinterest = pd.DataFrame(data=d2)

In [41]:
print(df_mG_genesofinterest)

                                           gene_group  mG_FS851  mG_FS852  \
0                           ech hydrogenase (echABCE)     204.0     284.0   
1        energy-converting hydrogenase A (ehaBCEGNOP)     323.0     497.0   
2      energy-converting hydrogenase B (ehbAHFIJKLNO)     168.0     209.0   
3                 coenzyme F420 hydrogenase (frhABDG)     475.0     765.0   
4                      hydrogenase (hyaABC/hybO/hybC)     205.0     258.0   
5                          [NiFe] hydrogenase (hydAB)      96.0     114.0   
6           methane/ammonia monooxygenase (pmoA-amoA)       1.0       0.0   
7              methyl-coenzyme M reductase (mcrABCDG)       8.0       9.0   
8                      nitrous-oxide reductase (nosZ)      21.0      28.0   
9   nitrite reductase (NO-forming)/hydroxylamine r...      87.0     122.0   
10                      nitric oxide reductase (norB)      33.0      24.0   
11        nitrite reductase (cytochrome c-552) (nrfA)      32.0      60.0   

In [42]:
print(df_mT_genesofinterest)

                                           gene_group  mT_FS851   mT_FS852  \
0                           ech hydrogenase (echABCE)  100962.0   340946.0   
1        energy-converting hydrogenase A (ehaBCEGNOP)  171602.0   842943.0   
2      energy-converting hydrogenase B (ehbAHFIJKLNO)   79310.0   294356.0   
3                 coenzyme F420 hydrogenase (frhABDG)  239701.0  1306331.0   
4                      hydrogenase (hyaABC/hybO/hybC)  145278.0   503106.0   
5                          [NiFe] hydrogenase (hydAB)   58884.0   171342.0   
6           methane/ammonia monooxygenase (pmoA-amoA)     168.0       23.0   
7              methyl-coenzyme M reductase (mcrABCDG)   17970.0   142682.0   
8                      nitrous-oxide reductase (nosZ)    1797.0    16234.0   
9   nitrite reductase (NO-forming)/hydroxylamine r...   11566.0    81117.0   
10                      nitric oxide reductase (norB)   25267.0    50165.0   
11        nitrite reductase (cytochrome c-552) (nrfA)   14697.0 

In [54]:
#divide each line in df_mG_genesofinterest by sccog_dict to get normalized read abundance
print(sccog_dict)

df_norma_mG_abun = pd.DataFrame()
df_norma_mG_abun['gene_group'] = df_mG_genesofinterest['gene_group']

df_norma_mG_abun['mG_FS851'] = df_mG_genesofinterest['mG_FS851'].div(sccog_dict['mG_FS851_sccogs_avg'])
df_norma_mG_abun['mG_FS852'] = df_mG_genesofinterest['mG_FS852'].div(sccog_dict['mG_FS852_sccogs_avg'])
df_norma_mG_abun['mG_FS854'] = df_mG_genesofinterest['mG_FS854'].div(sccog_dict['mG_FS854_sccogs_avg'])
df_norma_mG_abun['mG_FS856'] = df_mG_genesofinterest['mG_FS856'].div(sccog_dict['mG_FS856_sccogs_avg'])

print(df_norma_mG_abun)

{'mG_FS851_sccogs_avg': 48.77142857142857, 'mG_FS852_sccogs_avg': 56.0, 'mG_FS854_sccogs_avg': 237.94285714285715, 'mG_FS856_sccogs_avg': 180.6}
                                           gene_group  mG_FS851   mG_FS852  \
0                           ech hydrogenase (echABCE)  4.182777   5.071429   
1        energy-converting hydrogenase A (ehaBCEGNOP)  6.622730   8.875000   
2      energy-converting hydrogenase B (ehbAHFIJKLNO)  3.444640   3.732143   
3                 coenzyme F420 hydrogenase (frhABDG)  9.739309  13.660714   
4                      hydrogenase (hyaABC/hybO/hybC)  4.203281   4.607143   
5                          [NiFe] hydrogenase (hydAB)  1.968366   2.035714   
6           methane/ammonia monooxygenase (pmoA-amoA)  0.020504   0.000000   
7              methyl-coenzyme M reductase (mcrABCDG)  0.164030   0.160714   
8                      nitrous-oxide reductase (nosZ)  0.430580   0.500000   
9   nitrite reductase (NO-forming)/hydroxylamine r...  1.783831   2.178571 

In [58]:
#divide each line in df_mGT_genesofinterest by mT_readhits_dict to get normalized read abundance
print(mT_readhits_dict)

df_norma_mT_abun = pd.DataFrame()
df_norma_mT_abun['gene_group'] = df_mT_genesofinterest['gene_group']

df_norma_mT_abun['mT_FS851'] = df_mT_genesofinterest['mT_FS851'].div(mT_readhits_dict['mT_FS851_readhits'])
df_norma_mT_abun['mT_FS852'] = df_mT_genesofinterest['mT_FS852'].div(mT_readhits_dict['mT_FS852_readhits'])
df_norma_mT_abun['mT_FS854'] = df_mT_genesofinterest['mT_FS854'].div(mT_readhits_dict['mT_FS854_readhits'])
df_norma_mT_abun['mT_FS856'] = df_mT_genesofinterest['mT_FS856'].div(mT_readhits_dict['mT_FS856_readhits'])

print(df_norma_mT_abun)

{'mT_FS851_readhits': 436525353.0, 'mT_FS852_readhits': 1915431988.0, 'mT_FS854_readhits': 282876734.0, 'mT_FS856_readhits': 812137068.0}
                                           gene_group      mT_FS851  \
0                           ech hydrogenase (echABCE)  2.312855e-04   
1        energy-converting hydrogenase A (ehaBCEGNOP)  3.931089e-04   
2      energy-converting hydrogenase B (ehbAHFIJKLNO)  1.816848e-04   
3                 coenzyme F420 hydrogenase (frhABDG)  5.491113e-04   
4                      hydrogenase (hyaABC/hybO/hybC)  3.328054e-04   
5                          [NiFe] hydrogenase (hydAB)  1.348925e-04   
6           methane/ammonia monooxygenase (pmoA-amoA)  3.848574e-07   
7              methyl-coenzyme M reductase (mcrABCDG)  4.116599e-05   
8                      nitrous-oxide reductase (nosZ)  4.116599e-06   
9   nitrite reductase (NO-forming)/hydroxylamine r...  2.649560e-05   
10                      nitric oxide reductase (norB)  5.788209e-05   
11        

In [59]:
#write out dataframes to tsv files
df_norma_mG_abun.to_csv('normalized_mG_abun.tsv', sep="\t")
df_norma_mT_abun.to_csv('normalized_mT_abun.tsv', sep="\t")